# Sparkle V2 — Fase 5b: Diagnóstico del baseline

El LSTM v1 dio **36.9% de accuracy en Test**, peor que el baseline trivial de predecir siempre la clase mayoritaria (**45.6%**). Este notebook diagnostica por qué antes de seguir ajustando a ciegas.

**Qué hace:**
1. Confirma el baseline trivial (predecir siempre "Atento").
2. Prueba un modelo **no temporal** simple (estadísticas resumen por clip + Regresión Logística / Random Forest) — si esto tampoco supera el baseline, el problema es de *features*, no de arquitectura.
3. Si las features sí tienen señal, entrena una versión **ajustada** del LSTM: class weighting más suave, features de velocidad de cambio entre frames, learning rate más bajo con scheduler.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt

PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
DATASET_DIR = f'{PROJECT_DIR}/dataset_final'
MODELS_DIR = f'{PROJECT_DIR}/checkpoints'
os.makedirs(MODELS_DIR, exist_ok=True)

CLASS_NAMES = ['Atento', 'Desatencion_leve', 'Desatencion_marcada']

In [ ]:
def load_split(split_name):
    data = np.load(f'{DATASET_DIR}/{split_name}_consolidado.npz', allow_pickle=True)
    return {
        'X': data['X'],
        'boredom': data['boredom'],
    }

train = load_split('train')
val = load_split('validation')
test = load_split('test')

def collapse_boredom(boredom_array):
    return np.where(boredom_array == 0, 0, np.where(boredom_array == 1, 1, 2)).astype(np.int32)

y_train = collapse_boredom(train['boredom'])
y_val = collapse_boredom(val['boredom'])
y_test = collapse_boredom(test['boredom'])

print('Shapes:', train['X'].shape, val['X'].shape, test['X'].shape)

## 1. Baseline trivial (clase mayoritaria)

In [ ]:
majority_class = pd.Series(y_train).value_counts().idxmax()
trivial_preds = np.full_like(y_test, majority_class)
trivial_acc = accuracy_score(y_test, trivial_preds)
print(f'Clase mayoritaria (Train): {CLASS_NAMES[majority_class]}')
print(f'Accuracy del baseline trivial en Test: {trivial_acc:.4f}')
print('\nCualquier modelo que propongamos tiene que superar claramente este número para ser útil.')

## 2. Modelo no temporal: ¿las features tienen señal?

En vez de darle al modelo la secuencia completa, le damos estadísticas resumen por clip: media, desvío estándar, mínimo y máximo de cada feature a lo largo de los 60 frames. Esto colapsa la dimensión temporal — si un modelo simple con esto ya separa las clases razonablemente, confirma que las features (EAR, yaw, pitch) cargan señal real. Si ni así funciona, el problema es de features, no de arquitectura.

In [ ]:
def summarize_clip(X):
    # X: (n_clips, n_frames, n_features) -> (n_clips, n_features*4)
    mean = X.mean(axis=1)
    std = X.std(axis=1)
    minimum = X.min(axis=1)
    maximum = X.max(axis=1)
    return np.concatenate([mean, std, minimum, maximum], axis=1)

X_train_summary = summarize_clip(train['X'])
X_val_summary = summarize_clip(val['X'])
X_test_summary = summarize_clip(test['X'])

scaler_summary = StandardScaler()
X_train_summary_scaled = scaler_summary.fit_transform(X_train_summary)
X_val_summary_scaled = scaler_summary.transform(X_val_summary)
X_test_summary_scaled = scaler_summary.transform(X_test_summary)

print('Shape de features resumen:', X_train_summary_scaled.shape)

In [ ]:
print('--- Regresión Logística ---')
logreg = LogisticRegression(class_weight='balanced', max_iter=1000, multi_class='multinomial')
logreg.fit(X_train_summary_scaled, y_train)
y_pred_logreg = logreg.predict(X_test_summary_scaled)
print(f'Accuracy: {accuracy_score(y_test, y_pred_logreg):.4f}')
print(classification_report(y_test, y_pred_logreg, target_names=CLASS_NAMES, digits=3))

In [ ]:
print('--- Random Forest ---')
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_summary_scaled, y_train)
y_pred_rf = rf.predict(X_test_summary_scaled)
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(classification_report(y_test, y_pred_rf, target_names=CLASS_NAMES, digits=3))

In [ ]:
# Importancia de features del Random Forest — ayuda a ver qué variables cargan señal real
feature_labels = []
for stat in ['mean', 'std', 'min', 'max']:
    for f in ['ear_avg', 'ear_left', 'ear_right', 'yaw', 'pitch']:
        feature_labels.append(f'{f}_{stat}')

importances = pd.Series(rf.feature_importances_, index=feature_labels).sort_values(ascending=False)
print('Top 10 features más importantes según Random Forest:')
print(importances.head(10))

## Punto de decisión

Mirá los resultados de arriba antes de seguir a la sección 3:

- **Si Regresión Logística o Random Forest superan claramente el baseline trivial (45.6%)** → las features tienen señal, el problema del LSTM v1 era de entrenamiento (class weight agresivo, LR alto, arquitectura). Seguí a la sección 3, donde ajustamos el LSTM.
- **Si ninguno de los dos supera ~45-48%** → el problema es de features: EAR + yaw + pitch no alcanzan para distinguir bien el aburrimiento en este dataset. En ese caso, antes de reintentar el LSTM conviene: (a) sumar features nuevas (ver Apéndice), o (b) recién ahí evaluar sumar otro dataset como hablamos.

## 3. LSTM ajustado

Cambios respecto de la v1:
- **Features de velocidad de cambio** (delta frame-a-frame) para yaw y pitch, agregadas a la secuencia — capturan movimiento/inquietud, no solo posición estática.
- **Class weight suavizado** (raíz cuadrada de los pesos balanceados) para no sobrecorregir.
- **Learning rate más bajo** (3e-4 en vez de 1e-3) + `ReduceLROnPlateau` para evitar las oscilaciones que vimos en las curvas.
- Menos capacidad (menos unidades) para reducir el overfitting temprano que se vio en la v1.

In [ ]:
def add_delta_features(X):
    """X: (n_clips, n_frames, n_features) con columnas [ear_avg, ear_left, ear_right, yaw, pitch].
    Agrega delta frame-a-frame de yaw y pitch (columnas 3 y 4)."""
    deltas = np.diff(X[:, :, 3:5], axis=1, prepend=X[:, :1, 3:5])  # mismo largo, primer delta = 0
    return np.concatenate([X, deltas], axis=2)

X_train_v2 = add_delta_features(train['X'])
X_val_v2 = add_delta_features(val['X'])
X_test_v2 = add_delta_features(test['X'])

n_features_v2 = X_train_v2.shape[2]
n_frames = X_train_v2.shape[1]
print('Nuevo shape de features (con deltas de yaw/pitch):', X_train_v2.shape)

In [ ]:
scaler_v2 = StandardScaler()
X_train_v2_flat = X_train_v2.reshape(-1, n_features_v2)
scaler_v2.fit(X_train_v2_flat)

def scale_X_v2(X):
    shape = X.shape
    X_flat = X.reshape(-1, n_features_v2)
    return scaler_v2.transform(X_flat).reshape(shape)

X_train_v2 = scale_X_v2(X_train_v2)
X_val_v2 = scale_X_v2(X_val_v2)
X_test_v2 = scale_X_v2(X_test_v2)

In [ ]:
class_weights_array = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=y_train)
class_weights_soft = np.sqrt(class_weights_array)  # suavizado
class_weight_dict_v2 = {i: w for i, w in enumerate(class_weights_soft)}
print('Class weights v1 (balanced completo):', dict(enumerate(class_weights_array.round(3))))
print('Class weights v2 (suavizado, raíz cuadrada):', dict(enumerate(class_weights_soft.round(3))))

In [ ]:
def build_model_v2(n_frames, n_features, n_classes=3):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(n_frames, n_features)),
        tf.keras.layers.LSTM(32, return_sequences=True, recurrent_dropout=0.1),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.LSTM(16),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(n_classes, activation='softmax'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

model_v2 = build_model_v2(n_frames, n_features_v2)
model_v2.summary()

In [ ]:
checkpoint_path_v2 = f'{MODELS_DIR}/lstm_boredom_v2_best.keras'

callbacks_v2 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path_v2, monitor='val_loss', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
]

history_v2 = model_v2.fit(
    X_train_v2, y_train,
    validation_data=(X_val_v2, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict_v2,
    callbacks=callbacks_v2,
    verbose=1,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_v2.history['loss'], label='train')
axes[0].plot(history_v2.history['val_loss'], label='val')
axes[0].set_title('Loss (v2)')
axes[0].legend()
axes[1].plot(history_v2.history['accuracy'], label='train')
axes[1].plot(history_v2.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy (v2)')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
test_loss_v2, test_acc_v2 = model_v2.evaluate(X_test_v2, y_test, verbose=0)
print(f'Test accuracy (v2): {test_acc_v2:.4f}  |  Baseline trivial: {trivial_acc:.4f}')

y_pred_v2 = np.argmax(model_v2.predict(X_test_v2), axis=1)
print('\nReporte de clasificación (v2):')
print(classification_report(y_test, y_pred_v2, target_names=CLASS_NAMES, digits=3))

## Checkpoint de la Fase 5b

Pegame acá, en este orden:
1. Accuracy del baseline trivial (sección 1).
2. Accuracy y reporte de Regresión Logística y Random Forest (sección 2) — **esto es lo más importante**, define el diagnóstico.
3. Si corriste la sección 3: accuracy y reporte del LSTM v2.

Con eso vemos si el camino es seguir afinando el LSTM, sumar features, o evaluar otro dataset.